In [6]:
import requests
from bs4 import BeautifulSoup
import csv
from tqdm import tqdm

In [18]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36'
}
root_url = 'https://aromo.ru/perfumes/g-male/page/'

In [23]:
url_list = []
with tqdm(total=20) as pbar:
    for i in range(21, 40):
        page_url = f"{root_url}{i}/"

        page_response = requests.get(page_url, headers=headers)
        if page_response.status_code != 200:
            print(f"Failed to fetch: {page_url}, {page_response.status_code}")

        soup = BeautifulSoup(page_response.text, 'html.parser')

        urls_items = soup.find_all('a', class_='block-offer-item__link block-offer-item__link_full')

        for url_item in urls_items:
            url = url_item.get('href')
            url_list.append('https://aromo.ru' + url)

        pbar.update(1)

print(len(url_list))

 95%|█████████▌| 19/20 [00:36<00:01,  1.94s/it]

912


In [21]:
def process_note_name(note_name):
    note_name = note_name.get_text().strip().replace('\n', '').lower()
    first_bracket_index = note_name.find('(')
    if first_bracket_index == -1:
        return note_name
    note_name = note_name[:first_bracket_index]
    return note_name

In [27]:
with open('parfum_dataset_male.csv', mode='a', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    with tqdm(total=912) as pbar:
        for url in url_list:
            pbar.update(1)
            page_response = requests.get(url, headers=headers)
            if page_response.status_code != 200:
                print(f"Failed to fetch: {url}, {page_response.status_code}")

                continue

            soup = BeautifulSoup(page_response.text, 'html.parser')

            parfum_notes_list = [process_note_name(note) for note in soup.find_all('figcaption', class_='base-note__name')]
            if parfum_notes_list is None or parfum_notes_list == []:
                continue
            writer.writerow([' '.join(parfum_notes_list), 'm'])


100%|██████████| 912/912 [28:36<00:00,  1.88s/it]
